In [57]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd
from datetime import datetime
import os
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

In [165]:
data = pd.read_csv(r'..\data\train.csv')
test = pd.read_csv(r'..\data\test.csv')


In [199]:
sp_cols = ['RoomService','FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

def feat_eng(df):
    df['VIP'] = df['VIP'].astype(bool)
    df[['FN', 'LN']] = df['Name'].str.split(' ', expand=True)
    df['Family'] = df['LN'].duplicated(keep=False).map({True: 'family', False: 'not family'})
    df['sp_cols'] = (df[sp_cols] != 0).sum(axis=1)
    df[['deck', 'room', 'side']] = df['Cabin'].str.split('/', expand=True)
    
    # Define a nested function to calculate the rate
    try:
        vip = (df['VIP']==True).sum()
        transp = (df['sp_cols']==True).sum()

        df['Rate'] = np.where((df.get('VIP', 0) == True) & (df.get('sp_cols', 0) >=2),  
                            1 - (vip / transp), vip / transp)
    except:
        pass        
    return df

In [200]:
numeric_transformer = Pipeline([
                ('imputer', SimpleImputer(strategy='mean')),
                ('scaler', StandardScaler())
            ])
            

categorical_cols = ['HomePlanet','CryoSleep', 'Destination', 'VIP', 
                    'Family', 'deck', 'side'
                    ]
num_cols = ['Age','RoomService', 'FoodCourt', 'ShoppingMall', 'Spa','VRDeck','sp_cols','room','Rate']
# Create a preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
y_train = y_train.to_numpy()
# Fit and transform the training data
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# Create the DMatrix for XGBoost
dtrain = xgb.DMatrix(X_train_preprocessed, label=y_train)
dtest = xgb.DMatrix(X_test_preprocessed, label=y_test)

In [201]:
data = feat_eng(data)
test = feat_eng(test)   

In [203]:
#data = data.fillna(0)
X = data.drop(columns=['Transported','PassengerId','Name', 'FN', 'LN', 'Cabin'])  # Remove the target column
y = data['Transported']  # Extract the target column


In [204]:
#data preprocessing


In [205]:
# Initialize Optuna study
study = optuna.create_study(direction='maximize')

# Define the optimization function directly
def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': ['error', 'auc'],
        'tree_method': 'gpu_hist',
        'device': 'cuda',
        'verbosity':0,
        
        # Parameters to optimize
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'eta': trial.suggest_float('eta', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.3, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'seed': 42
    }
    
    # Cross validation
    cv_scores = []
    kf = StratifiedKFold(n_splits=5, 
                         shuffle=True, 
                         random_state=42)
    
    for train_idx, val_idx in kf.split(X_train_preprocessed, y_train):
        X_fold_train, X_fold_val = X_train_preprocessed[train_idx], X_train_preprocessed[val_idx]
        y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
    
    # Create DMatrix for XGBoost
        dtrain = xgb.DMatrix(X_fold_train, label=y_fold_train)
        dval = xgb.DMatrix(X_fold_val, label=y_fold_val)
        
        # Train model
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=1000,
            early_stopping_rounds=20,
            evals=[(dval, 'val')],
            verbose_eval=False
        )
        
        # Get validation score
        val_pred = model.predict(dval)
        score = roc_auc_score(y_fold_val, val_pred)
        cv_scores.append(score)
    
    return np.mean(cv_scores)

# Run optimization
study.optimize(objective, n_trials=5, show_progress_bar=True)

# Get best parameters
best_params = study.best_params

# Add fixed parameters to best_params
best_params.update({
    'objective': 'binary:logistic',
    'eval_metric': ['error', 'auc'],
    'tree_method': 'gpu_hist',
    'device': 'cuda',
    'seed': 42
})

# Print results
print("\nBest parameters:", best_params)
print(f"Best CV score: {study.best_value:.4f}")

# Train final model with best parameters
dtrain = xgb.DMatrix(X_train_preprocessed, label=y_train)
dval = xgb.DMatrix(X_test_preprocessed, label=y_test)

final_model = xgb.train(
    best_params,
    dtrain,
    num_boost_round=10000,
    early_stopping_rounds=50,
    evals=[(dval, 'validation')],
    verbose_eval=100
)

[I 2025-02-24 11:47:19,723] A new study created in memory with name: no-name-4318283c-2acf-4ee3-9736-87f5a9cb9b09
c:\Users\Leo\.conda\envs\tensor4\lib\site-packages\optuna\progress_bar.py:49: ExperimentalWarning: Progress bar is experimental (supported from v1.2.0). The interface can change in the future.
  self._init_valid()
 20%|██        | 1/5 [00:06<00:24,  6.01s/it]

[I 2025-02-24 11:47:25,732] Trial 0 finished with value: 0.8981250829206029 and parameters: {'max_depth': 10, 'eta': 0.05013151600639796, 'subsample': 0.4324179551228604, 'colsample_bytree': 0.9756031769866302, 'min_child_weight': 6, 'lambda': 0.00530826228318881}. Best is trial 0 with value: 0.8981250829206029.


 40%|████      | 2/5 [00:08<00:10,  3.66s/it]

[I 2025-02-24 11:47:27,746] Trial 1 finished with value: 0.8985697569759672 and parameters: {'max_depth': 3, 'eta': 0.10299395152615815, 'subsample': 0.6969913599806792, 'colsample_bytree': 0.41968358053266364, 'min_child_weight': 4, 'lambda': 2.0362301731319664e-05}. Best is trial 1 with value: 0.8985697569759672.


 60%|██████    | 3/5 [00:09<00:05,  2.80s/it]

...

 80%|████████  | 4/5 [00:17<00:04,  4.69s/it]

[I 2025-02-24 11:47:37,115] Trial 3 finished with value: 0.9004182434315198 and parameters: {'max_depth': 4, 'eta': 0.0234626289748023, 'subsample': 0.8576182235568732, 'colsample_bytree': 0.6801523343425877, 'min_child_weight': 1, 'lambda': 0.1309203006114687}. Best is trial 3 with value: 0.9004182434315198.


100%|██████████| 5/5 [00:34<00:00,  6.83s/it]

[I 2025-02-24 11:47:53,877] Trial 4 finished with value: 0.8902286181697244 and parameters: {'max_depth': 10, 'eta': 0.05115276697919467, 'subsample': 0.43969453058720215, 'colsample_bytree': 0.3034450027533601, 'min_child_weight': 1, 'lambda': 8.26682264966392e-07}. Best is trial 3 with value: 0.9004182434315198.

Best parameters: {'max_depth': 4, 'eta': 0.0234626289748023, 'subsample': 0.8576182235568732, 'colsample_bytree': 0.6801523343425877, 'min_child_weight': 1, 'lambda': 0.1309203006114687, 'objective': 'binary:logistic', 'eval_metric': ['error', 'auc'], 'tree_method': 'gpu_hist', 'device': 'cuda', 'seed': 42}
Best CV score: 0.9004
[0]	validation-error:0.25819	validation-auc:0.82087


[100]	validation-error:0.21334	validation-auc:0.87901
[200]	validation-error:0.20644	validation-auc:0.88933
[300]	validation-error:0.20011	validation-auc:0.89377
[400]	validation-error:0.20299	validation-auc:0.89531
[500]	validation-error:0.20011	validation-auc:0.89657
[600]	validation-error:0.20011	validation-auc:0.89754
[700]	validation-error:0.19897	validation-auc:0.89799
[800]	validation-error:0.19781	validation-auc:0.89823
[900]	validation-error:0.19667	validation-auc:0.89846
[1000]	validation-error:0.19839	validation-auc:0.89887
[1027]	validation-error:0.19839	validation-auc:0.89871


In [206]:
test

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,VRDeck,Name,FN,LN,Family,sp_cols,deck,room,side,Rate
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,...,0.0,Nelly Carsoning,Nelly,Carsoning,family,0,G,3,S,0.605072
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,...,0.0,Lerome Peckers,Lerome,Peckers,not family,2,F,4,S,0.605072
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,...,0.0,Sabih Unhearfus,Sabih,Unhearfus,not family,0,C,0,S,0.605072
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,...,585.0,Meratz Caltilter,Meratz,Caltilter,not family,3,C,1,S,0.605072
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,...,0.0,Brence Harperez,Brence,Harperez,family,2,F,5,S,0.605072
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4272,9266_02,Earth,True,G/1496/S,TRAPPIST-1e,34.0,False,0.0,0.0,0.0,...,0.0,Jeron Peter,Jeron,Peter,family,0,G,1496,S,0.605072
4273,9269_01,Earth,False,NaN,TRAPPIST-1e,42.0,False,0.0,847.0,17.0,...,144.0,Matty Scheron,Matty,Scheron,family,4,NaN,NaN,NaN,0.605072
4274,9271_01,Mars,True,D/296/P,55 Cancri e,NaN,False,0.0,0.0,0.0,...,0.0,Jayrin Pore,Jayrin,Pore,family,0,D,296,P,0.605072
4275,9273_01,Europa,False,D/297/P,NaN,NaN,False,0.0,2680.0,0.0,...,523.0,Kitakan Conale,Kitakan,Conale,family,2,D,297,P,0.605072


In [207]:
blind = test.drop(columns=['PassengerId','Name', 'FN','LN','Cabin',]) 
blind_preprocessed = preprocessor.transform(blind)
dblind = xgb.DMatrix(blind_preprocessed)
y_pred_prob = final_model.predict(dblind)
# Calculate the accuracy
predicted_y  = (y_pred_prob  > 0.5).astype(int)

sub = pd.merge(test[['PassengerId']], pd.DataFrame(predicted_y), left_index=True, right_index=True)#.to_csv('data/ak_submission.csv', index=False)
sub = sub.rename(columns={0:'Transported'})
sub['Transported'] = sub['Transported'].map({0:False, 1:True})
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"submissionxgboost_{timestamp}.csv"
filepath = os.path.join(r"../submissions", filename)  # Cross-platform

# Save DataFrame to CSV
sub.to_csv(filepath, index=False)